# Word Mover's Distance Model Notebook
The word mover's distance model [http://proceedings.mlr.press/v37/kusnerb15.pdf] can be used to measure the distance between a new list of words and an existing database of words. This is done by embedding each word and then calculating the distance from every word in this embedding to all of the embeddings of known senteneces in a database. We elect to use this model in an attempt to map comment data onto known comments to classify them for their cognitive presence level.

## Setup 

First we are going to import nltk's english stopwords and define how we will process sentences

In [2]:
# Import and download stopwords from NLTK.
from nltk.corpus import stopwords
from nltk import download
download('stopwords')  # Download stopwords list.
stop_words = stopwords.words('english')

def preprocess(sentence):
    return [w for w in sentence.lower().split() if w not in stop_words]



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\tehho\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Next we will import a word2vec embedding to create the embedding space that all of our comments will be mapped on

In [3]:
# Set up gensim
import gensim.downloader as api
model = api.load('word2vec-google-news-300')


Now we will read in the combined phases data that should be saved in the .data folder relative to the root directory of this project

In [4]:
# Get data
import pandas as pd
import numpy as np
import os
data_dir = '.data'

pdf_comments = pd.read_csv(os.path.join(data_dir, 'Combined Phases.csv'))
# Apply preprocessing
pdf_comments['embedding'] = pdf_comments['Text'].apply(lambda x: preprocess(x))
pdf_comments = pdf_comments.drop('Text', axis=1)

pdf_comments.shape

(583, 2)

## Modelling

First we will split the data into test and train. For the word movers distance, we are approximately using a knn model so we will just use the training data to set up the database that the new sentences (test) will be compared to. We elect to use a test size of .2 because we have many classes and there is not a lot of data to speak of

In [5]:
# Function to split dataset
from sklearn.model_selection import train_test_split

X = pdf_comments['embedding']
y = pdf_comments['CP Score']
# Split before we impute or min_max scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=0)

len(y_test)

117

For our KNN Model, we will neeed a funciton to break ties if there are any number of classes equally represented in the K neartest neightbors.

In [6]:
# What is a better voting model?
def voter(n):
    return 5 - n


#### KNN Model
Here we implement a very simple KNN model, not that we are just returning the top value if there is one then use the voting model. This isn't always the best method, we could just use our voting model for all

In [7]:
# Build knn model
def lazy_knn(n_neighbors, text, X_database, y_database, dist_func=model.wmdistance, vote=voter):
    # Get distance metric
    dists = X_database.apply(lambda x: dist_func(x, text))

    # Get top n matching classes
    top_n = dists.sort_values()[:n_neighbors]
    top_n_classes = y_database[top_n.index]

    # We can go with outright winner
    counts = top_n_classes.value_counts()

    if len(counts) == 0:
        return -1

    if len(counts) == 1:
        return counts.index[0]

    if counts.iloc[0] != counts.iloc[1]:
        return counts.index[0]

    # Voting method
    vote_values = top_n.apply(lambda x: voter(x))
    vote_values.index = top_n_classes
    election_results = vote_values.groupby(level=0).sum().sort_values(ascending=False)
    return election_results.index[0]


Here we test how well this model performs on the test data. Values for N, and the voting methods are both things we can vary to make this work

In [8]:

n_neighbors = 7
X_database = X_train
y_database = y_train
vote = voter
x_0 = X_test.values[1]

# hard test
y_pred = [lazy_knn(n_neighbors, text, X_database, y_database) for text in X_test.values]

from sklearn import metrics
# Print the confusion matrix
print(metrics.confusion_matrix(y_test, y_pred))

# Print the precision and recall, among other metrics
print(metrics.classification_report(y_test, y_pred, digits=3))

# Cohen Kappa
print(metrics.cohen_kappa_score(y_test, y_pred))

# Explicit F1
print(metrics.f1_score(y_test, y_pred, average='weighted'))


[[27  4  2  1  1]
 [ 4  6  6  0  4]
 [ 8  1  3  4  3]
 [ 3  4  3  7  3]
 [ 5  6  6  4  2]]
              precision    recall  f1-score   support

           0      0.574     0.771     0.659        35
           1      0.286     0.300     0.293        20
           2      0.150     0.158     0.154        19
           3      0.438     0.350     0.389        20
           4      0.154     0.087     0.111        23

    accuracy                          0.385       117
   macro avg      0.320     0.333     0.321       117
weighted avg      0.350     0.385     0.360       117

0.20715294117647065
0.36033204513504696
